# Imports

In [2]:
import pandas as pd
import numpy as np

# Ruta al archivo en donde se encuentra el tráfico benigno de prueba

In [6]:
file_path = 'data/Monday-WorkingHours.pcap_ISCX.csv'
print(f"Cargando tráfico benigno base desde: {file_path}...")
df = pd.read_csv(file_path)
print("Tráfico benigno base cargado exitosamente.")

Cargando tráfico benigno base desde: data/Monday-WorkingHours.pcap_ISCX.csv...
Tráfico benigno base cargado exitosamente.


# 1. Limpieza de nombres de columnas
# Elimina espacios en blanco al principio y al final de los nombres de todas las columnas.

In [7]:
df.columns = df.columns.str.strip()

# 2. Tratamiento de valores infinitos y nulos.
# Acá limpiamos los valores - infinito y + infinito para cambiarlos por NaN, así vamos a estandarizar los valores corruptos

In [8]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
529913,443,18738,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
529914,53,60797,2,2,80,156,40,40,40.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
529915,53,154,2,2,64,96,32,32,32.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
529916,53,155,2,2,80,144,40,40,40.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


# Eliminamos todas las filas que contengan algún NaN

In [9]:
df.dropna(inplace=True)

# 3. Verificación final de la forma y los datos

In [10]:
print(f"Limpieza completada exitosamente.")
print(f"Forma final del DataFrame: {df.shape[0]} filas y {df.shape[1]} columnas.\n")
print("Primeras 5 filas del dataset limpio:")

Limpieza completada exitosamente.
Forma final del DataFrame: 529481 filas y 79 columnas.

Primeras 5 filas del dataset limpio:


# Visualizamos la celda en Jupyter

In [11]:
display(df.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


# Importamos scikit-learn para el procesamiento de Machine Learning.
# Con esto vamos a procesar el medio millón de filas de nuestro dataset de forma casi instantánea y sin errores de cálculo manual

In [12]:
from sklearn.preprocessing import MinMaxScaler

# 1. Verificamos la columna de etiquetas, suele llamarse 'Label'

In [13]:
if 'Label' in df.columns:
    print("Distribución de etiquetas en este archivo:")
    print(df['Label'].value_counts())
    
    # Separamos las características (X) de las etiquetas (y)
    X_raw = df.drop(columns=['Label'])
    y_labels = df['Label']
else:
    print("No se encontró la columna 'Label'. Verificá los nombres:", df.columns)
    X_raw = df

Distribución de etiquetas en este archivo:
Label
BENIGN    529481
Name: count, dtype: int64


# 2. Hacemos una normalización Min-Max (escala de 0 a 1) 
# Las redes neuronales son sensibles a las diferencias de magnitud (ej. `Flow Duration` en millones vs `Flags` en 0 a 1). Elegimos `MinMaxScaler` porque nuestra arquitectura de Autoencoder va a usar una función de activación *Sigmoide* en su capa de salida (que solo devuelve valores entre 0 y 1). Esto garantiza que la red pueda calcular correctamente el error de reconstrucción.

In [14]:
print("\nIniciando escalado de características (MinMaxScaler)...")
scaler = MinMaxScaler()


Iniciando escalado de características (MinMaxScaler)...


# Entrenamos el scaler y transformamos los datos en un solo paso

In [15]:
X_scaled = scaler.fit_transform(X_raw)

# Convertimos el array de numpy de vuelta a un DataFrame para visualizarlo aesthetic

In [16]:
df_scaled = pd.DataFrame(X_scaled, columns=X_raw.columns)
print("Escalado completado exitosamente.")

Escalado completado exitosamente.


# Primeras 5 líneas del escalado

In [17]:
display(df_scaled.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,0.750561,4.166667e-08,0.000005,0.0,0.000009,0.0,0.000257,0.002617,0.001293,0.0,...,0.000005,0.999999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.750561,1.666667e-08,0.000005,0.0,0.000009,0.0,0.000257,0.002617,0.001293,0.0,...,0.000005,0.999999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.750561,1.666667e-08,0.000005,0.0,0.000009,0.0,0.000257,0.002617,0.001293,0.0,...,0.000005,0.999999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.750561,1.666667e-08,0.000005,0.0,0.000009,0.0,0.000257,0.002617,0.001293,0.0,...,0.000005,0.999999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.755108,3.333334e-08,0.000005,0.0,0.000009,0.0,0.000257,0.002617,0.001293,0.0,...,0.000005,0.999999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
